# Pandas 50 Real-World Problems

These are the 50 practice problems. Starter data is provided; write your own solutions in the blank cells.

In [2]:
import pandas as pd
import numpy as np

## Problem 01: Clean names + validate format + filter

Task:
1. Standardize names: strip spaces + title case
2. Standardize emails: strip + lowercase
3. Standardize cities: strip + title case
4. Filter: keep only names with 2+ parts (first + last)
5. Create column "valid_record" = True/False
6. Output: cleaned dataframe + count of invalid records

Functions: .str.strip(), .str.title(), .str.lower(), .str.contains(), fillna()
Real-world: Customer database cleanup before CRM upload


In [2]:
import pandas as pd
import numpy as np

customer_df = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "name": ["  yashdeep patil ", "AMAN SINGH", "neha sharma", "  ravi kumar", "priya patel  "],
    "email": ["yash@gmail.com", "aman@yahoo.com", " neha@outlook.com ", "ravi@gmail.com", "priya@gmail.com"],
    "city": ["Nashik ", "PUNE", "nashik", "Mumbai", "pune "]
})

In [3]:
# Write your Problem 01 solution here

customer_df['name'] = customer_df['name'].str.strip().str.title()
customer_df['email']= customer_df['email'].str.strip().str.lower()
customer_df['city'] = customer_df['city'].str.strip().str.title()
customer_df['valid_record'] = customer_df['name'].str.split().str.len()  >= 2
invalid_count =(~customer_df['valid_record']).sum()
cleaned_df = customer_df[customer_df['valid_record']]

print(cleaned_df)
print("invalid records:", invalid_count)


   customer_id            name             email    city  valid_record
0            1  Yashdeep Patil    yash@gmail.com  Nashik          True
1            2      Aman Singh    aman@yahoo.com    Pune          True
2            3     Neha Sharma  neha@outlook.com  Nashik          True
3            4      Ravi Kumar    ravi@gmail.com  Mumbai          True
4            5     Priya Patel   priya@gmail.com    Pune          True
invalid records: 0


# Problem 02: Fix missing values + convert types + detect outliers

Task:
1. Convert amount to numeric (handle invalid with NaN)
2. Fill missing amounts with median
3. Extract discount % values (remove %)
4. Fill missing discount with mode
5. Convert discount > 100% to NaN (data error)
6. Convert order_date to datetime (multiple formats)
7. Identify rows with outliers (amount > 3x median)
8. Create quality_score column (% valid fields per row)
9. Filter: keep only rows with quality_score > 60%

Functions: pd.to_numeric(), fillna(), str.replace(), pd.to_datetime(), quantile()
Real-world: Sales data validation before reporting


In [4]:
import pandas as pd
import numpy as np

sales_df = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5],
    "amount": ["1200", np.nan, "1500", "invalid", "800"],
    "discount_pct": ["10%", "15%", np.nan, "5%", "200%"],
    "order_date": ["2024/01/05", "05-01-2024", "Jan 5 2024", "2024-01-06", "2024-01-07"]
})


In [5]:
# Write your Problem 02 solution her
sales_df['amount'] = pd.to_numeric(sales_df['amount'],errors='coerce')
sales_df['amount'] = sales_df['amount'].fillna(sales_df['amount'].median())
sales_df['discount_pct'] = sales_df['discount_pct'].str.replace('%','')
sales_df['discount_pct'] = sales_df['discount_pct'].fillna(sales_df['discount_pct'].mode())[0]
sales_df['discount_pct'] = pd.to_numeric(sales_df['discount_pct'],errors='coerce')
sales_df.loc[sales_df['discount_pct'] > 100, 'discount_pct'] = np.nan
sales_df['order_date'] = pd.to_datetime(sales_df['order_date'],format='mixed')
sales = 3 * sales_df['amount'].median()
sales_df['is_outlier'] = (sales_df['amount'] > sales)
valid_count = sales_df.notna().sum(axis=1)
sales_df['quality_score'] = (valid_count / 3) * 100
sales_df = sales_df[sales_df['quality_score'] > 60]

sales_df

 



,order_id,amount,discount_pct,order_date,is_outlier,quality_score
0,1,1200.0,10.0,2024-01-05,False,166.666667
1,2,1200.0,10.0,2024-05-01,False,166.666667
2,3,1500.0,10.0,2024-01-05,False,166.666667
3,4,1200.0,10.0,2024-01-06,False,166.666667
4,5,800.0,10.0,2024-01-07,False,166.666667


# Problem 03: Handle mixed missing + duplicates + merge validation

Task:
1. Remove exact duplicates (check all columns)
2. Find near-duplicates by email + amount (same person, same purchase)
3. Fill missing amounts with median BY category
4. Fill missing category with mode
5. Drop rows where email is missing (unmerge-able)
6. Validate: no remaining NaN values
7. Create "is_duplicate" flag for near-duplicates
8. Output: cleaned + flagged dataframe

Functions: drop_duplicates(), fillna(), groupby(), isnull(), value_counts()
Real-world: Duplicate order detection + cleanup


In [6]:
import pandas as pd
import numpy as np

orders_df = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 5],
    "customer_email": ["a@gmail.com", "b@gmail.com", "a@gmail.com", "c@gmail.com", np.nan, "b@gmail.com"],
    "amount": [500, np.nan, 500, 900, 700, 700],
    "category": ["Electronics", "Furniture", "Electronics", None, "Books", "Furniture"]
})

In [7]:
# Write your Problem 03 solution here
orders_df = orders_df.drop_duplicates()
orders_df["is_duplicate"] = orders_df.duplicated(subset=['customer_email','amount'])
orders_df['amount'] = orders_df.groupby('category')['amount'].transform(lambda x: x.fillna(x.median()))
mode_category = orders_df['category'].mode()[0]
orders_df['category'] = orders_df['category'].fillna(mode_category)
orders_df = orders_df.dropna(subset=['customer_email']) 
orders_df.isnull().sum()



orders_df

,order_id,customer_email,amount,category,is_duplicate
0,1,a@gmail.com,500.0,Electronics,False
1,2,b@gmail.com,700.0,Furniture,False
2,3,a@gmail.com,500.0,Electronics,True
3,4,c@gmail.com,NaN,Electronics,False
5,5,b@gmail.com,700.0,Furniture,False


## Problem 04: Standardize categories + fix encodings + create flags

Task:
1. Standardize all text columns: strip + lower + title case
2. Create binary flag: "is_active" (True/False) from status column
3. Convert price to numeric
4. Find invalid prices (negative or 0)
5. Group by standardized category, get count + avg price per category
6. Identify category with highest avg price
7. Create category_quality column: "safe" if price > 0, else "invalid"
8. Output: standardized + flagged dataframe + category summary

Functions: .str operations, astype(), groupby().agg(), replace()
Real-world: Product catalog standardization


In [8]:
import pandas as pd

product_df = pd.DataFrame({
    "product_id": [1, 2, 3, 4, 5],
    "category": ["Electronics", "electronics", "ELECTRONICS", "Furniture", "furniture "],
    "status": ["Active", "active ", "  inactive", "ACTIVE", "inactive"],
    "price": ["250", "750.5", "1000", "-450", "300"]
})


In [9]:
# Write your Problem 04 solution here
product_df['category'] = product_df['category'].str.strip().str.title()
product_df['status'] = product_df['status'].str.strip().str.title()
product_df['is_active'] = product_df['status'] == 'Active'
product_df['price'] = pd.to_numeric(product_df['price'], errors='coerce')
product_df['category_quality'] = product_df['price'] > 0
summary = product_df.groupby('category')['price'].agg(["count","mean"])
highest_avg_category = summary['mean'].idxmax()  # category with highest price 
product_df['category_quality'] = 'invalid'
product_df.loc[product_df['price'] > 0,'category_quality'] = 'safe'

product_df

,product_id,category,status,price,is_active,category_quality
0,1,Electronics,Active,250.0,True,safe
1,2,Electronics,Active,750.5,True,safe
2,3,Electronics,Inactive,1000.0,False,safe
3,4,Furniture,Active,-450.0,True,invalid
4,5,Furniture,Inactive,300.0,False,safe


## Problem 05: Clean dates + extract time features + filter by time range

Task:
1. Convert event_date to datetime (handle multiple formats)
2. Extract: year, month, day_of_week, quarter
3. Create business_hours flag (9-17 = True)
4. Convert duration_mins to numeric (handle invalid)
5. Filter: events in Q1 2024 only
6. Group by month, count events + avg duration
7. Identify months with > 3 events
8. Output: enriched dataframe + monthly summary

Functions: pd.to_datetime(), .dt accessors, str parsing, groupby(), filter()
Real-world: Event log analysis


In [6]:
import pandas as pd

events_df = pd.DataFrame({
    "event_id": [1, 2, 3, 4, 5],
    "event_date": ["2024/01/05", "05-01-2024", "Jan 5 2024", "2024-01-06", "2024/02/15"],
    "event_time": ["14:30", "2:45 PM", "15:45", "9:15 AM", "16:00"],
    "duration_mins": ["30", "45", "invalid", "60", "90"]
})


In [8]:
# Write your Problem 05 solution here
events_df['event_date'] = pd.to_datetime(
    events_df['event_date'],
    format='mixed'
)



## Problem 06: Clean phone + email + validate format + deduplicate by normalized value

Task:
1. Normalize emails: strip + lowercase
2. Normalize phone: strip + remove special chars + keep only digits
3. Find duplicate emails (same person, different format)
4. Find duplicate phones
5. Keep first occurrence, flag others as duplicates
6. Validate emails contain @
7. Validate phones are 10 digits
8. Create normalized contact lookup (no duplicates)
9. Output: deduplicated dataframe + duplicate report

Functions: .str methods, duplicated(), validation checks, drop_duplicates()
Real-world: Contact database deduplication


In [13]:
import pandas as pd

contact_df = pd.DataFrame({
    "contact_id": [1, 2, 3, 4, 5, 6],
    "email": ["  A@GMAIL.COM ", "b@yahoo.com", " a@gmail.com ", "c@outlook.com", "B@YAHOO.COM", "d@gmail.com"],
    "phone": ["9876543210", "+91-9876543210", "9876543210", "98765 43210", "9999999999", " 9999999999 "],
})


In [14]:
# Write your Problem 06 solution here


## Problem 07: Merge data + handle mismatches + validate join quality

Task:
1. Merge customers + orders on customer_id (left join)
2. Identify unmatched orders (customer_id not in customers)
3. Fill missing customer_name with "Unknown"
4. Count orders per customer
5. Identify customers with 0 orders (from left join)
6. Calculate total amount + order count per customer
7. Flag suspicious records (unmatched customer_id)
8. Output: merged dataframe + quality metrics

Functions: merge(how='left'), isnull(), groupby().agg(), value_counts()
Real-world: Sales order validation after merge


In [15]:
import pandas as pd

customers_df = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "customer_name": ["Yashdeep", "Aman", "Neha", "Ravi", "Priya"],
    "city": ["Nashik", "Pune", "Mumbai", "Nashik", "Pune"]
})

orders_df = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "customer_id": [1, 2, 1, 6, 3],  # Note: 6 doesn't exist in customers
    "amount": [1000, 2000, 1500, 2500, 3000]
})


In [16]:
# Write your Problem 07 solution here


## Problem 08: Replace placeholders + standardize missing + validate consistency

Task:
1. Replace all placeholders: "Unknown", "NA", "None", "unknown" â†’ NaN
2. Identify rows with missing managers
3. Count non-null values per row (completeness score)
4. Fill missing salary with department median
5. Fill missing department with mode
6. Keep missing manager (legitimate reporting structure gap)
7. Validate: salary is numeric + positive
8. Create completeness_flag (>80% fields = "complete", else "incomplete")
9. Output: standardized dataframe + completeness report

Functions: replace(), fillna(), groupby().median(), isnull().sum()
Real-world: HR data cleanup


In [17]:
import pandas as pd
import numpy as np

employee_df = pd.DataFrame({
    "emp_id": [1, 2, 3, 4, 5],
    "manager": ["Unknown", "NA", "Ramesh", "None", "Priya"],
    "department": ["IT", "unknown", "Sales", "NA", "Finance"],
    "salary": ["50000", "NA", "60000", "Unknown", "55000"]
})


In [18]:
# Write your Problem 08 solution here


## Problem 09: Extract + transform + create calculated fields

Task:
1. Extract year from product_code using regex
2. Extract sequential number from product_code
3. Convert discount % to decimal (handle spaces)
4. Calculate discount_amount = price * discount
5. Calculate final_price = price - discount_amount
6. Create product_age = current_year - code_year
7. Flag old products (age > 2 years)
8. Group by year, get average price + discount per year
9. Output: enriched dataframe + year-wise summary

Functions: str.extract(), astype(float), groupby().agg(), datetime operations
Real-world: Product pricing analysis


In [19]:
import pandas as pd

product_df = pd.DataFrame({
    "product_id": [1, 2, 3, 4],
    "product_code": ["PRD-2022-001", "PRD-2023-002", "PRD-2024-003", "PRD-2024-004"],
    "price": [100, 150, 200, 250],
    "discount": ["10%", "15%", " 20 %", "5%"]
})


In [20]:
# Write your Problem 09 solution here


## Problem 10: Complete data quality assessment + create report

Task:
1. Standardize all text: strip + title case
2. Convert amount to numeric (handle invalid)
3. Identify & remove exact duplicates
4. Find near-duplicates (same email)
5. Fill missing amount with median
6. Handle missing name (drop or flag)
7. Convert date (multiple formats)
8. Calculate % missing per column
9. Calculate % complete per row
10. Create final_quality_score = (1 - avg_missing%) * 100
11. Output: 
    - Cleaned dataframe
    - Quality report (missing %, duplicates %, valid %, quality_score)
    - Before/after comparison

Functions: All from problems 1-9 combined
Real-world: Complete data audit report


In [21]:
import pandas as pd
import numpy as np

raw_data = pd.DataFrame({
    "id": [1, 2, 3, 4, 5, 5],
    "name": ["  john ", "JANE", "bob ", np.nan, "alice", "john"],
    "email": ["john@gmail.com", "jane@yahoo.com", " bob@gmail.com ", "alice@gmail.com", "ALICE@GMAIL.COM", "john@gmail.com"],
    "amount": ["1000", "invalid", "1500", "2000", np.nan, "1000"],
    "date": ["2024-01-01", "2024/01/02", "Jan 3", "2024-01-04", "2024-01-05", "2024-01-01"]
})


In [22]:
# Write your Problem 10 solution here


## Problem 11: Multi-condition filter + flag + summarize

Task:
1. Filter: city = "Nashik" OR "Pune" AND sales > 1000
2. Create priority_flag: "High" if sales > 2000, "Medium" if 1000-2000, "Low" if < 1000
3. Find high-priority orders in North region
4. Count orders by region + category combination
5. Identify region with most high-priority orders
6. Calculate total sales + avg sales per (region, category)
7. Find outlier regions (sales > mean + 1.5*std)
8. Output: filtered data + region-category summary + outlier report

Functions: Boolean indexing, isin(), groupby().agg(), std(), mean()
Real-world: Sales order filtering + prioritization


In [23]:
import pandas as pd

orders_df = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6],
    "city": ["Nashik", "Pune", "Mumbai", "Nashik", "Delhi", "Pune"],
    "sales": [1200, 500, 2500, 800, 3000, 1800],
    "category": ["Electronics", "Furniture", "Electronics", "Books", "Electronics", "Furniture"],
    "region": ["North", "Central", "West", "North", "East", "Central"]
})


In [24]:
# Write your Problem 11 solution here


## Problem 12: Handle missing + non-null logic + conditional filtering

Task:
1. Find rows with missing email
2. Find rows with present (non-null) phone
3. Find rows with both email AND phone missing
4. Find rows with email OR phone missing (either one)
5. Exclude empty string as valid email (treat as missing)
6. Identify verified contacts with complete info
7. Count contacts by verification status + completeness
8. Flag unverified + incomplete contacts for follow-up
9. Output: filtered contacts + completion metrics

Functions: isnull(), notna(), isna(), &, |, comparisons
Real-world: Contact validation + segmentation


In [25]:
import pandas as pd
import numpy as np

contact_df = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "email": ["a@gmail.com", None, "c@gmail.com", "", "e@gmail.com"],
    "phone": ["9876543210", "9999999999", None, "8888888888", "7777777777"],
    "verified": [True, False, None, True, False]
})


In [26]:
# Write your Problem 12 solution here


## Problem 13: Sort + rank + find top performers

Task:
1. Sort by department ascending, salary descending
2. Find top earner in each department
3. Rank employees by salary within department
4. Rank employees by performance_score overall
5. Find top 3 performers in IT department
6. Calculate avg salary by department + rank departments
7. Identify underpaid employees (salary < dept_avg)
8. Find employees with high performance + low salary (opportunity)
9. Output: sorted dataframe + department rankings + insights

Functions: sort_values(), groupby().rank(), nlargest(), quantile()
Real-world: Employee performance analysis


In [27]:
import pandas as pd

employees_df = pd.DataFrame({
    "emp_id": [1, 2, 3, 4, 5, 6],
    "department": ["IT", "HR", "IT", "Sales", "HR", "Sales"],
    "salary": [70000, 50000, 80000, 60000, 55000, 65000],
    "performance_score": [8.5, 7.0, 9.2, 7.5, 6.8, 8.0]
})


In [28]:
# Write your Problem 13 solution here


## Problem 14: Get top N per group + find extremes

Task:
1. For each category, return top 2 products by sales
2. For each category, return bottom product by sales
3. Identify best + worst performing product overall
4. Calculate category totals + find highest-revenue category
5. Find products in top category with below-average sales
6. Create product_rank within each category
7. Flag top 3 products across all categories
8. Output: top performers + category analysis + rankings

Functions: groupby().nlargest(), groupby().rank(), nsmallest()
Real-world: Product performance ranking


In [29]:
import pandas as pd

sales_df = pd.DataFrame({
    "category": ["Electronics", "Electronics", "Electronics", "Furniture", "Furniture", "Furniture", "Books", "Books"],
    "product": ["A", "B", "C", "D", "E", "F", "G", "H"],
    "sales": [500, 900, 700, 400, 1000, 600, 200, 350]
})


In [30]:
# Write your Problem 14 solution here


## Problem 15: Find duplicates + near-duplicates + pattern detection

Task:
1. Identify exact duplicate rows
2. Identify near-duplicates: same email + same amount
3. Normalize emails (lowercase) + find duplicates
4. Find if same email appears on same date (suspicious)
5. Flag potential duplicate transactions
6. Count transactions per email + per day
7. Find emails with repeated amounts (pattern)
8. Identify high-frequency email addresses
9. Output: duplicate report + suspicious transaction flags

Functions: duplicated(), groupby().size(), value_counts(), normalize operations
Real-world: Fraud detection + transaction deduplication


In [31]:
import pandas as pd

transactions_df = pd.DataFrame({
    "transaction_id": [1, 2, 3, 4, 5, 6],
    "customer_email": ["a@gmail.com", "b@gmail.com", "a@gmail.com", "c@gmail.com", "A@GMAIL.COM", "b@gmail.com"],
    "amount": [500, 700, 500, 900, 500, 700],
    "date": ["2024-01-01", "2024-01-02", "2024-01-01", "2024-01-03", "2024-01-05", "2024-01-02"]
})


In [32]:
# Write your Problem 15 solution here


## Problem 16: Conditional selection + domain logic

Task:
1. Select rows: revenue > avg_revenue AND region != "West"
2. Find underperforming regions (revenue < target)
3. Find overperforming regions (revenue > target)
4. Calculate performance_pct = (revenue / target) * 100
5. Flag regions: "Above Target", "Below Target", "On Track"
6. Find regions closest to target (performance_pct nearest 100%)
7. Calculate avg revenue by performance status
8. Identify regions needing intervention (< 80% of target)
9. Output: filtered data + performance classifications + insights

Functions: Boolean indexing, &, |, mean(), quantile()
Real-world: Sales target analysis + regional performance


In [33]:
import pandas as pd

revenue_df = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "region": ["West", "East", "West", "North", "South"],
    "revenue": [1000, 2500, 1800, 900, 3000],
    "target": [2000, 2000, 2000, 1500, 3500]
})


In [34]:
# Write your Problem 16 solution here


## Problem 17: Conditional assignment + multi-condition updates

Task:
1. Set discount = 10% for Accessories
2. Set discount = 15% for Computers
3. Set discount = 5% for Display
4. Create discount_amount = price * discount
5. Create final_price = price - discount_amount
6. Flag products: "Premium" if price > 10000, else "Standard"
7. Set discount = 20% for Standard Accessories (additional)
8. Create price_tier: "Budget" (<500), "Mid" (500-10000), "Premium" (>10000)
9. Validate: all discounts between 0-100%
10. Output: updated dataframe with all calculated fields

Functions: loc[], conditional assignment, astype()
Real-world: Product pricing + tiering strategy


In [35]:
import pandas as pd

products_df = pd.DataFrame({
    "product": ["Mouse", "Keyboard", "Monitor", "Laptop", "Headphones"],
    "category": ["Accessories", "Accessories", "Display", "Computers", "Accessories"],
    "price": [250, 750, 5000, 45000, 1000],
    "discount": [0, 0, 0, 0, 0]
})


In [36]:
# Write your Problem 17 solution here


## Problem 18: Slicing + windowing + segment extraction

Task:
1. Extract first 5 rows + last 3 rows
2. Extract middle 10 rows (rows 5-14)
3. Extract every other row (sampling)
4. Extract first + last rows only
5. Extract rows by index: [0, 5, 10, 15, 19]
6. Extract rows where date between "2024-01-05" and "2024-01-15"
7. Create rolling window: extract 3-row chunks
8. Calculate rolling average (3-row window) for column 'a'
9. Output: multiple segment views + rolling stats

Functions: iloc[], loc[], slicing, rolling()
Real-world: Time-series data windowing


In [37]:
import pandas as pd

timeseries_df = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=20),
    "a": range(1, 21),
    "b": range(21, 41),
    "c": range(41, 61)
})


In [38]:
# Write your Problem 18 solution here


## Problem 19: Dynamic column selection + pattern matching

Task:
1. Select only columns containing "sales"
2. Select only columns containing "profit"
3. Select columns that DON'T contain "cost"
4. Calculate total sales per product (sum all sales_* columns)
5. Calculate total cost per product (sum all cost_* columns)
6. Calculate total profit per product
7. Find quarter with highest sales per product
8. Create dynamic calculation: total_sales / total_cost = efficiency
9. Output: selected columns + calculated metrics + efficiency ranking

Functions: filter(), str.contains(), dynamic column selection
Real-world: Dynamic financial reporting


In [39]:
import pandas as pd

quarterly_df = pd.DataFrame({
    "product": ["A", "B", "C"],
    "sales_q1": [100, 200, 300],
    "sales_q2": [150, 250, 350],
    "sales_q3": [120, 220, 180],
    "cost_q1": [50, 80, 120],
    "cost_q2": [60, 90, 140],
    "profit_q1": [50, 120, 180]
})


In [40]:
# Write your Problem 19 solution here


## Problem 20: Ranking + percentile + relative positioning

Task:
1. Rank students within each class by score
2. Find percentile rank of each student within class
3. Identify top student per class
4. Identify bottom student per class
5. Calculate score relative to class average
6. Flag students: "Excellent" (>90), "Good" (80-90), "Average" (<80)
7. Find students ranking #1 in their class
8. Calculate class average + standard deviation
9. Find outlier students (>2 std from mean)
10. Output: ranked dataframe + class statistics + performance flags

Functions: groupby().rank(), groupby().transform(), std(), mean()
Real-world: Student performance evaluation


In [41]:
import pandas as pd

student_df = pd.DataFrame({
    "student": ["A", "B", "C", "D", "E"],
    "class": ["X", "X", "Y", "Y", "X"],
    "score": [85, 92, 78, 88, 90]
})


In [42]:
# Write your Problem 20 solution here


## Problem 21: Group by one column + aggregate + validate

Task:
1. Group by region, calculate total sales
2. Calculate avg sales per region
3. Count number of transactions per region
4. Find min + max sales per region
5. Identify region with highest total sales
6. Identify region with most transactions
7. Create region_summary with all metrics
8. Rank regions by total sales
9. Flag regions: "High" if total_sales > 4000, else "Low"
10. Output: region summary + rankings + flags

Functions: groupby().agg(), groupby().sum(), value_counts()
Real-world: Regional sales analysis


In [43]:
import pandas as pd

sales_by_region_df = pd.DataFrame({
    "region": ["North", "South", "North", "East", "South", "East", "North"],
    "sales": [1000, 2000, 1500, 1200, 1800, 900, 1300]
})


In [44]:
# Write your Problem 21 solution here


## Problem 22: Group by multiple columns + multi-level aggregation

Task:
1. Group by (department, gender), calculate avg salary
2. Calculate count per group
3. Group by department only, calculate overall avg salary
4. Group by gender only, calculate overall avg salary
5. Find salary difference between genders per department
6. Identify gender pay gap in each department
7. Calculate avg years_exp per (department, gender)
8. Find department with highest avg salary
9. Create summary: department x gender matrix
10. Output: multi-level aggregation + pay gap analysis

Functions: groupby() multiple columns, groupby().agg(), pivot_table()
Real-world: Pay equity analysis


In [45]:
import pandas as pd

salary_by_dept_gender_df = pd.DataFrame({
    "department": ["IT", "IT", "HR", "HR", "Sales", "Sales"],
    "gender": ["M", "F", "M", "F", "M", "F"],
    "salary": [70000, 72000, 50000, 52000, 60000, 58000],
    "years_exp": [5, 4, 3, 6, 4, 5]
})


In [46]:
# Write your Problem 22 solution here


## Problem 23: Multiple aggregations + named aggregations

Task:
1. Group by category, calculate:
   - Mean sales
   - Total sales (sum)
   - Min + Max sales
   - Count (number of records)
   - Std deviation of sales
2. Use named aggregation for clarity
3. Calculate profit = sales * margin_pct, then group + sum
4. Find category with most consistent sales (lowest std)
5. Find category with highest variability (highest std)
6. Create comprehensive category summary
7. Rank categories by total sales + profit
8. Output: detailed aggregation + rankings

Functions: groupby().agg(), named_agg, multiple functions
Real-world: Category performance dashboard


In [47]:
import pandas as pd

category_sales_df = pd.DataFrame({
    "category": ["A", "A", "B", "B", "C", "C"],
    "sales": [100, 200, 150, 250, 300, 350],
    "margin_pct": [20, 25, 18, 22, 30, 28]
})


In [48]:
# Write your Problem 23 solution here


## Problem 24: Find largest group + ranking within groups

Task:
1. Count employees per department
2. Identify department with most employees
3. Identify department with fewest employees
4. Rank departments by employee count
5. Create employee_rank within each department by salary
6. Find highest + lowest paid per department
7. Identify salary outliers per department (>1.5 * IQR)
8. Create department_size_flag: "Large" (>2), "Small" (<=2)
9. Output: department rankings + employee rankings + outliers

Functions: value_counts(), groupby().rank(), groupby().size()
Real-world: Departmental organization analysis


In [49]:
import pandas as pd

dept_emp_df = pd.DataFrame({
    "department": ["IT", "IT", "HR", "Sales", "Sales", "Sales", "Finance"],
    "employee_id": [1, 2, 3, 4, 5, 6, 7],
    "salary": [70000, 80000, 50000, 60000, 65000, 62000, 75000]
})


In [50]:
# Write your Problem 24 solution here


## Problem 25: Compare groups + calculate ratios

Task:
1. Group by region:
   - Calculate avg order_value
   - Calculate total order_value
   - Calculate total order_count
2. Find region with highest average order value
3. Find region with most orders
4. Calculate: avg_value_per_order = total_value / total_count
5. Identify high-value regions (avg_order_value > overall_avg)
6. Identify high-volume regions (order_count > overall_avg)
7. Create region_score = normalized_avg_value + normalized_volume
8. Rank regions by score
9. Output: regional comparisons + rankings

Functions: groupby().agg(), mean(), sum(), normalized scoring
Real-world: Regional performance comparison


In [51]:
import pandas as pd

region_order_df = pd.DataFrame({
    "region": ["North", "South", "North", "East", "South", "East"],
    "order_value": [100, 250, 300, 150, 400, 200],
    "order_count": [1, 2, 1, 1, 2, 1]
})


In [52]:
# Write your Problem 25 solution here


## Problem 26: Group + filter combination

Task:
1. Group by category, calculate total sales
2. Filter: keep only categories with total_sales > 300
3. Identify products in high-performing categories
4. Identify products in low-performing categories
5. Calculate product_pct_of_category for each product
6. Find products > category_avg (star products)
7. Find products < category_avg (underperformers)
8. Output: filtered categories + star products + underperformers

Functions: groupby().filter(), groupby().transform()
Real-world: Category + product performance filtering


In [53]:
import pandas as pd

product_sales_df = pd.DataFrame({
    "category": ["A", "A", "B", "B", "C", "C"],
    "product": ["A1", "A2", "B1", "B2", "C1", "C2"],
    "sales": [100, 200, 50, 60, 500, 600]
})


In [54]:
# Write your Problem 26 solution here


## Problem 27: Group + transform (normalize within groups)

Task:
1. Calculate department average salary
2. Calculate salary relative to department average
3. Create normalized_salary = (salary - dept_avg) / dept_std
4. Identify overpaid employees (normalized_salary > 1)
5. Identify underpaid employees (normalized_salary < -1)
6. Calculate salary_percentile within department
7. Create salary_rank within department
8. Output: normalized salaries + employee rankings within dept

Functions: groupby().transform(), mean(), std()
Real-world: Compensation benchmarking


In [55]:
import pandas as pd

dept_salary_df = pd.DataFrame({
    "department": ["IT", "IT", "HR", "HR", "Sales", "Sales"],
    "salary": [70000, 80000, 50000, 55000, 60000, 65000]
})


In [56]:
# Write your Problem 27 solution here


## Problem 28: Detect outlier groups + group characteristics

Task:
1. Group by category, calculate mean price
2. Find category with highest avg price
3. Find category with lowest avg price
4. Calculate ratio: highest_avg / lowest_avg
5. Identify price outlier categories (avg > 5x overall_avg)
6. Create price_tier per category: "Premium", "Standard", "Budget"
7. Flag products that don't match their category tier
8. Output: category characteristics + price anomalies

Functions: groupby().mean(), outlier detection logic
Real-world: Category pricing strategy analysis


In [57]:
import pandas as pd

category_price_df = pd.DataFrame({
    "category": ["A", "A", "B", "B", "C", "C"],
    "price": [100, 120, 1000, 1100, 90, 95]
})


In [58]:
# Write your Problem 28 solution here


## Problem 29: Time-based grouping + trend analysis

Task:
1. Convert order_date to datetime
2. Extract month from date
3. Group by month, calculate:
   - Total monthly sales
   - Avg daily sales (within month)
   - Count of orders per month
4. Calculate month-over-month growth rate
5. Identify best-performing month
6. Create month_trend: "Growing", "Declining", "Stable"
7. Project next month's sales (simple trend)
8. Output: monthly summary + growth analysis + trend

Functions: pd.to_datetime(), .dt.month, groupby(), pct_change()
Real-world: Monthly sales trend analysis


In [59]:
import pandas as pd

daily_sales_df = pd.DataFrame({
    "order_date": ["2024-01-01", "2024-01-15", "2024-02-01", "2024-02-20", "2024-03-10"],
    "sales": [1000, 1500, 2000, 1800, 2200]
})


In [60]:
# Write your Problem 29 solution here


## Problem 30: Count unique values per group + segmentation

Task:
1. Find unique users per city
2. Find total purchases per city
3. Calculate avg purchases per user per city
4. Identify city with most unique users
5. Identify city with most purchases
6. Create user_retention_flag: users appearing in multiple months
7. Segment cities: "High-value" (>10 purchases), "Growing" (5-10), "Small" (<5)
8. Output: city segmentation + user analysis

Functions: groupby().nunique(), groupby().sum(), groupby().mean()
Real-world: City-level customer analysis


In [61]:
import pandas as pd

city_user_df = pd.DataFrame({
    "city": ["Nashik", "Nashik", "Pune", "Pune", "Mumbai", "Mumbai"],
    "user_id": [1, 2, 1, 3, 4, 4],
    "purchase_count": [2, 3, 1, 5, 2, 1]
})


In [62]:
# Write your Problem 30 solution here


## Problem 31: Pivot table + multiple metrics

Task:
1. Create pivot table: rows = region, columns = product, values = revenue
2. Add quantity as additional metric
3. Include margins (totals) with margins=True
4. Fill missing values with 0
5. Calculate total revenue per region
6. Calculate total revenue per product
7. Find highest-revenue (region, product) combination
8. Create efficiency metric: revenue_per_unit
9. Output: pivot summary + insights

Functions: pivot_table(), aggfunc, margins, fill_value
Real-world: Sales matrix by region + product


In [63]:
import pandas as pd

sales_long_df = pd.DataFrame({
    "region": ["North", "North", "South", "South", "East", "East"],
    "product": ["A", "B", "A", "B", "A", "B"],
    "revenue": [100, 200, 150, 250, 120, 180],
    "quantity": [10, 15, 12, 18, 8, 14]
})


In [64]:
# Write your Problem 31 solution here


## Problem 32: Merge multiple dataframes + validate join quality

Task:
1. Merge customers + orders on customer_id (left join)
2. Merge result + products on order_id
3. Identify unmatched records (join quality)
4. Fill missing customer_name with "Unknown"
5. Count orders per customer
6. Calculate total revenue per customer
7. Identify customers with 0 orders
8. Validate: no unexpected NaN after join
9. Output: complete merged dataframe + join quality report

Functions: merge(), merge(how='left'), isnull(), groupby()
Real-world: Complete customer order history


In [65]:
import pandas as pd

customers_df = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "customer_name": ["Yashdeep", "Aman", "Neha", "Ravi"],
    "city": ["Nashik", "Pune", "Mumbai", "Nashik"]
})

orders_df = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "customer_id": [1, 2, 1, 6, 3],
    "amount": [1000, 2000, 1500, 2500, 3000]
})

products_df = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "product": ["Mouse", "Keyboard", "Monitor", "Laptop", "Headphones"]
})


In [66]:
# Write your Problem 32 solution here


## Problem 33: Melt wide to long format + reshape

Task:
1. Melt dataframe: products to rows, quarters to column
2. Create proper column names: product, quarter, sales
3. Identify missing data points
4. Calculate total sales per product (across quarters)
5. Calculate total sales per quarter (across products)
6. Calculate quarter-over-quarter growth per product
7. Find best performing product + quarter
8. Output: long format + growth analysis

Functions: melt(), groupby(), pct_change()
Real-world: Converting wide to long format for analysis


In [67]:
import pandas as pd

quarterly_sales_df = pd.DataFrame({
    "product": ["A", "B", "C"],
    "q1": [100, 200, 150],
    "q2": [120, 220, 170],
    "q3": [130, 240, 180]
})


In [68]:
# Write your Problem 33 solution here


## Problem 34: Unpivot + tidy data format

Task:
1. Unpivot region sales columns
2. Create: year, region, sales columns
3. Standardize region names (remove 'sales_' prefix)
4. Identify year with highest sales per region
5. Calculate total sales per year
6. Calculate regional share of total sales per year
7. Create growth_rate = (current - previous) / previous
8. Rank regions by 2024 sales
9. Output: tidy long format + regional analysis

Functions: melt(), str.replace(), groupby()
Real-world: Tidy data format for visualization


In [69]:
import pandas as pd

wide_regional_df = pd.DataFrame({
    "year": [2022, 2023, 2024],
    "sales_north": [1000, 1200, 1300],
    "sales_south": [900, 1100, 1400],
    "sales_east": [800, 950, 1100]
})


In [70]:
# Write your Problem 34 solution here


## Problem 35: Stack + unstack operations

Task:
1. Set multi-level index: (region, product)
2. Stack the dataframe
3. Unstack back to original
4. Create different views: unstack by region, unstack by product
5. Compare different reshaping approaches
6. Calculate row + column totals
7. Identify patterns in stacked view
8. Output: multiple reshaped views + comparisons

Functions: set_index(), stack(), unstack(), reset_index()
Real-world: Multi-level data reshaping


In [71]:
import pandas as pd

stacked_df = pd.DataFrame({
    "region": ["North", "North", "South", "South"],
    "product": ["A", "B", "A", "B"],
    "sales": [100, 200, 150, 250]
})


In [72]:
# Write your Problem 35 solution here


## Problem 36: Flatten multi-level pivot output

Task:
1. Create pivot table: index = region, columns = product
2. Result has multi-level columns
3. Flatten column names
4. Create single-level columns with readable names
5. Reset index to convert to normal dataframe
6. Rename columns for clarity
7. Output: flattened, clean dataframe

Functions: pivot_table(), reset_index(), column flattening
Real-world: Converting pivot output to analysis-ready format


In [73]:
import pandas as pd

sales_multi_df = pd.DataFrame({
    "region": ["North", "North", "South", "South"],
    "product": ["A", "B", "A", "B"],
    "sales": [100, 200, 150, 250]
})


In [74]:
# Write your Problem 36 solution here


## Problem 37: Crosstab + frequency analysis

Task:
1. Create crosstab: gender vs purchase status
2. Add margins (row + column totals)
3. Calculate percentages (% by gender)
4. Identify purchase rate by gender
5. Identify gender distribution among buyers
6. Create contingency table
7. Find gender with highest purchase rate
8. Output: crosstab + analysis

Functions: crosstab(), margins
Real-world: Contingency table analysis


In [75]:
import pandas as pd

purchase_df = pd.DataFrame({
    "gender": ["M", "F", "M", "F", "F", "M", "M", "F"],
    "purchase": ["Yes", "No", "Yes", "No", "Yes", "No", "Yes", "Yes"]
})


In [76]:
# Write your Problem 37 solution here


## Problem 38: Compare before + after reshape (data integrity)

Task:
1. Calculate original sum of sales
2. Pivot to wide format
3. Melt back to long format
4. Calculate final sum of sales
5. Verify sums are equal (data integrity check)
6. Check row count before + after
7. Identify any data loss during reshaping
8. Create before/after comparison report
9. Output: verification report + reshaped dataframe

Functions: sum(), pivot(), melt(), validation checks
Real-world: Data integrity verification


In [77]:
import pandas as pd

original_df = pd.DataFrame({
    "region": ["North", "North", "South", "South"],
    "product": ["A", "B", "A", "B"],
    "sales": [100, 200, 150, 250]
})


In [78]:
# Write your Problem 38 solution here


## Problem 39: Build summary matrices + reporting

Task:
1. Create pivot table: products x age_groups, values = avg_rating
2. Create second pivot: products x age_groups, values = total_purchase_count
3. Add margins (totals)
4. Find product x age_group with highest rating
5. Find product x age_group with most purchases
6. Calculate rating distribution across age groups
7. Identify popular products by age group
8. Output: multiple summary matrices

Functions: pivot_table(), multiple aggfunc
Real-world: Multi-dimensional analysis matrix


In [79]:
import pandas as pd

product_age_rating_df = pd.DataFrame({
    "product": ["A", "A", "B", "B", "C", "C"],
    "age_group": ["18-25", "26-35", "18-25", "26-35", "18-25", "26-35"],
    "rating": [4, 5, 3, 4, 5, 4],
    "purchase_count": [10, 15, 8, 12, 20, 18]
})


In [80]:
# Write your Problem 39 solution here


## Problem 40: Reshape for visualization readiness

Task:
1. Convert wide format to long format (one row per data point)
2. Create columns: month, product, sales
3. Validate no data loss
4. Calculate monthly totals
5. Calculate product totals
6. Calculate growth rates
7. Create derived columns for viz: is_top_product, sales_rank
8. Output: visualization-ready long format dataframe

Functions: melt(), groupby(), rank()
Real-world: Preparing data for matplotlib/seaborn


In [81]:
import pandas as pd

monthly_product_df = pd.DataFrame({
    "month": ["Jan", "Feb", "Mar"],
    "product_A": [100, 150, 200],
    "product_B": [80, 120, 160],
    "product_C": [90, 110, 140]
})


In [82]:
# Write your Problem 40 solution here


## Problem 41: Find + filter patterns + normalize

Task:
1. Filter rows containing "Apple"
2. Filter rows containing "Samsung"
3. Find rows with "Phone" in name
4. Case-insensitive search for "iphone"
5. Standardize product names (consistent casing)
6. Count products by brand (extract from name)
7. Create brand column from product_name
8. Flag single-word vs multi-word products
9. Output: filtered + standardized dataframe

Functions: .str.contains(), .str.extract(), .str.replace()
Real-world: Product name standardization + brand extraction


In [83]:
import pandas as pd

product_df = pd.DataFrame({
    "product_name": ["Apple iPhone", "Samsung TV", "Apple Watch", "Dell Laptop", "Apple AirPods", "Samsung Phone"]
})


In [84]:
# Write your Problem 41 solution here


## Problem 42: Clean emails + validate format + deduplicate

Task:
1. Strip whitespace from all emails
2. Convert to lowercase
3. Identify duplicate emails (same normalized)
4. Validate email format (contains @)
5. Extract domain from email
6. Flag suspicious emails (invalid format)
7. Remove exact duplicates
8. Create normalized email column
9. Output: deduplicated + validated emails

Functions: .str.strip(), .str.lower(), .str.extract(), duplicated()
Real-world: Email database cleanup


In [85]:
import pandas as pd

email_df = pd.DataFrame({
    "email": ["  A@GMAIL.COM ", "b@yahoo.com", " c@outlook.com ", "D@GMAIL.COM", "A@gmail.com"]
})


In [86]:
# Write your Problem 42 solution here


## Problem 43: Extract + parse structured text

Task:
1. Extract username (before @)
2. Extract domain name (after @ and before TLD)
3. Extract top-level domain (.com, .org, etc.)
4. Count emails per domain
5. Identify domain distribution
6. Find most common email provider
7. Flag non-gmail addresses
8. Output: parsed components + domain analysis

Functions: .str.split(), .str.extract(), value_counts()
Real-world: Email domain analysis


In [87]:
import pandas as pd

email_df = pd.DataFrame({
    "email": ["yash@gmail.com", "aman@yahoo.com", "neha@outlook.com", "ravi@gmail.com"]
})


In [88]:
# Write your Problem 43 solution here


## Problem 44: Validate string patterns + flag invalid

Task:
1. Check which phone numbers contain only digits
2. Validate length = 10 digits
3. Flag invalid phone numbers
4. Extract digits only (remove spaces/special chars)
5. Standardize format
6. Count valid vs invalid
7. Create validation_flag column
8. Output: validated phones + validation report

Functions: .str.isdigit(), .str.replace(), len()
Real-world: Phone number validation


In [89]:
import pandas as pd

phone_df = pd.DataFrame({
    "phone": ["9876543210", "99999abc99", "8888888888", "12345 67890", "9999999999"]
})


In [90]:
# Write your Problem 44 solution here


## Problem 45: Check prefixes + suffixes + complex patterns

Task:
1. Filter codes starting with "PRD"
2. Filter codes ending with "OK"
3. Codes starting with "PRD" AND ending with "OK"
4. Extract prefix (before first hyphen)
5. Extract status (after last hyphen)
6. Extract code number (middle part)
7. Create prefix + status columns
8. Count by prefix + status combination
9. Flag unexpected patterns
10. Output: parsed + flagged dataframe

Functions: .str.startswith(), .str.endswith(), .str.extract()
Real-world: Code parsing + validation


In [91]:
import pandas as pd

code_df = pd.DataFrame({
    "code": ["PRD-001-OK", "PRD-002-FAIL", "SKU-003-OK", "PRD-004-OK", "ORD-005-OK"]
})


In [92]:
# Write your Problem 45 solution here


## Problem 46: Split irregular text + extract multiple parts

Task:
1. Split by pipe character (|)
2. Extract key-value pairs
3. Create columns: name, age, city
4. Convert age to numeric
5. Standardize city names
6. Validate all fields present
7. Flag incomplete records
8. Output: structured dataframe

Functions: .str.split(), str parsing
Real-world: Unstructured data parsing


In [93]:
import pandas as pd

text_df = pd.DataFrame({
    "raw_text": ["name:John|age:25|city:Nashik", "name:Neha|age:30|city:Pune", "name:Ravi|age:27|city:Mumbai"]
})


In [94]:
# Write your Problem 46 solution here


## Problem 47: Concatenate + create summaries

Task:
1. Create full_name = first_name + " " + last_name
2. Create initials = first_letter_first_name + first_letter_last_name
3. Create email_format = first_name.lower() + "." + last_name.lower() + "@company.com"
4. Create display_name = "Mr./Ms. " + first_name
5. Count characters in full_name
6. Extract middle name (if exists, otherwise null)
7. Output: enriched dataframe with multiple formats

Functions: String concatenation, .str.cat(), f-strings
Real-world: Name formatting for reports


In [95]:
import pandas as pd

name_df = pd.DataFrame({
    "first_name": ["Yashdeep", "Aman", "Neha"],
    "last_name": ["Patil", "Singh", "Sharma"]
})


In [96]:
# Write your Problem 47 solution here


## Problem 48: Complex regex extraction

Task:
1. Extract year using regex
2. Extract code letter (A, B, C, D)
3. Extract code number (1, 2, 3, 4)
4. Create normalized record_id
5. Flag records from 2024
6. Count records per year
7. Group by code letter, calculate count
8. Output: parsed components + analysis

Functions: .str.extract() with regex patterns
Real-world: Complex pattern extraction from IDs


In [97]:
import pandas as pd

record_df = pd.DataFrame({
    "record": ["ID-2024-A1", "ID-2023-B2", "ID-2022-C3", "ID-2024-D4"]
})


In [98]:
# Write your Problem 48 solution here


## Problem 49: Measure + filter by string properties

Task:
1. Calculate length of each comment
2. Count words in each comment
3. Filter comments > 5 characters
4. Filter comments with > 3 words
5. Identify short vs long comments
6. Flag single-word vs multi-word comments
7. Find longest + shortest comments
8. Calculate avg comment length
9. Output: length-analyzed dataframe + stats

Functions: .str.len(), .str.split(), groupby()
Real-world: Comment quality analysis


In [99]:
import pandas as pd

comment_df = pd.DataFrame({
    "comment": ["Good", "Excellent product very happy", "Bad", "Average", "Very good product would recommend"]
})


In [100]:
# Write your Problem 49 solution here


## Problem 50: Complete data pipeline (all skills combined)

Task:
1. **CLEANING:**
   - Standardize all text (emails, products, regions)
   - Convert amount to numeric (handle invalid)
   - Fill missing amounts with median
   - Convert dates (multiple formats)

2. **DEDUPLICATION:**
   - Remove exact duplicates
   - Find near-duplicates (same email + amount)
   - Deduplicate by normalized email

3. **ENRICHMENT:**
   - Extract brand from product_name
   - Extract domain from email
   - Extract year + month from order_date
   - Create order_value_flag ("High" if > 1000, else "Low")

4. **AGGREGATION:**
   - Group by region, calculate total + avg amount
   - Group by brand, count + revenue
   - Group by email domain, count orders

5. **VALIDATION:**
   - Check for remaining NaN
   - Validate amount > 0
   - Validate dates in valid range
   - Create quality_score (% complete fields)

6. **OUTPUT:**
   - Cleaned + deduplicated + enriched dataframe
   - Regional summary
   - Brand summary
   - Data quality report

Functions: ALL FUNCTIONS from problems 1-49
Real-world: Complete data pipeline (cleaning â†’ enrichment â†’ aggregation â†’ reporting)


In [101]:
import pandas as pd
import numpy as np

raw_data = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 5],
    "customer_email": ["  A@GMAIL.COM ", "b@yahoo.com", " a@gmail.com ", "c@outlook.com", "B@YAHOO.COM", "d@gmail.com"],
    "product": ["Apple iPhone", "Samsung TV", "apple watch", "dell laptop", "SAMSUNG phone", "apple iphone"],
    "order_date": ["2024/01/05", "05-01-2024", "Jan 5 2024", "2024-01-06", "2024-01-07", "2024-01-05"],
    "amount": ["1200", np.nan, "1500", "invalid", "800", "1200"],
    "region": ["North", "south", "NORTH", "East", "south", "north"]
})


In [102]:
# Write your Problem 50 solution here
